# Getting started

In this section, we will cover the very basics of creating a visualization graph using the `neo4j-viz` library.
We will use a small toy graph representing the purchase history of a few people and products.

We start by instantiating the `Nodes` and `Relationships` we want in our graph.
The only mandatory fields for a node are the "id", and "source" and "target" for a relationship.
But the other fields can optionally be used to customize the appearance of the nodes and relationships in the visualization.

Lastly we create a `VisualizationGraph` object with the nodes and relationships we created, and call its `render_widget` method to display the graph as an interactive widget.

In [1]:
from neo4j_viz import GraphSelection, Node, Relationship, VisualizationGraph

nodes = [
    Node(id=0, size=10, caption="Person"),
    Node(id=1, size=10, caption="Product"),
    Node(id=2, size=20, caption="Product"),
    Node(id=3, size=10, caption="Person"),
    Node(id=4, size=10, caption="Product"),
]
relationships = [
    Relationship(
        source=0,
        target=1,
        caption="BUYS",
    ),
    Relationship(
        source=0,
        target=2,
        caption="BUYS",
    ),
    Relationship(
        source=3,
        target=2,
        caption="BUYS",
    ),
]

VG = VisualizationGraph(nodes=nodes, relationships=relationships)

VG.color_nodes(field="size")
VG.set_node_captions(field="size")

widget = VG.render_widget(initial_zoom=2)
widget

As we can see in the graph above, the radius of one of the nodes is larger than the others.
This is because we set the "size" field of the node to 20, while the others are set to 10.

At this time all nodes have the same color.
If we want to distinguish between the different types of nodes, we can color them differently with the `color_nodes` method.
We can pass the field we want to use to color the nodes as an argument.
In this case, we will use the "caption" field.
Nodes with the same "caption" will have the same color.
We will use the default colorscheme, which is the Neo4j colorscheme.


We are now easily able to distinguish between the different types of nodes in the graph.

## Interacting with the widget

The `render_widget()` method we used above returns an interactive widget for Jupyter environments (JupyterLab, VS Code, Colab). It provides two-way sync between Python and JavaScript, so you can interact with the graph and update it programmatically.

If you instead need static HTML — for embedding in a document or a Streamlit app — use `render()`, which returns an `IPython.display.HTML` object.

Below we update the widget's data after displaying it. Changes sync automatically to the visualization above.

In [ ]:
# Run this cell multiple times - each run adds a new node to the widget above
import random

new_id = len(widget.nodes)
target_id = random.choice([n.id for n in widget.nodes])

new_node = Node(id=new_id, size=10, caption="Person")
new_rel = Relationship(source=new_id, target=target_id, caption="KNOWS")

widget.add_data(nodes=new_node, relationships=new_rel)

In [ ]:
widget.color_relationships(field="caption")

In [ ]:
widget.nodes[0].size = 50
widget.sync_nodes()  # manually trigger sync to update widget

In [ ]:
widget.set_zoom(1.5)  # change the rendering options dynamically

### Reacting to the selection

The widget exposes the current selection via its `selected` attribute, a typed `GraphSelection` with `nodeIds` and `relationshipIds`. To react to selection changes interactively, register a callback with `widget.on_selection_change(...)` - a convenience wrapper around `widget.observe(...)` whose callback receives the new `GraphSelection` directly and runs every time you select nodes or relationships in the widget above.

For example, the callback below connects each node you select to another node in the graph. Run the cell once to register it, then click a node in the widget above and watch a new `KNOWS` relationship appear automatically:

In [ ]:
# Run this cell once to register the callback, then select a node in the widget above.
def on_selection_change(selection: GraphSelection) -> None:
    # Selection IDs are strings, so match them against str(node.id) to recover the nodes.
    selected_ids = set(selection.nodeIds)
    selected_nodes = [n for n in widget.nodes if str(n.id) in selected_ids]
    if not selected_nodes:
        return

    source_node = selected_nodes[0]
    other_nodes = [n for n in widget.nodes if n.id != source_node.id]
    target_node = random.choice(other_nodes)
    widget.add_data(
        relationships=Relationship(
            source=source_node.id, target=target_node.id, caption="KNOWS"
        ),
    )


widget.on_selection_change(on_selection_change)